# Procesamiento y limpieza de los datos

In [1]:
# ============================================================
# Configuración inicial: imports y directorio de trabajo
# ============================================================
import datetime as dt
import pandas as pd
import pathlib
import os
import time
import holidays  # Calendario de festivos colombianos (ver celda "Variables de calendario")
# os.chdir(pathlib.Path.cwd())

# --- Bloque específico de Google Colab (desactivado para ejecución local) ---
# Conexión de google drive con el repo
#from google.colab import drive
#drive.mount('/content/drive')

###==========================
#ruta = os.path.join("/content/drive/MyDrive","MonografiaUdeA")
#os.chdir(ruta)

# Directorio de trabajo actual (carpeta donde vive este notebook). Todas las
# rutas de este notebook (datasets/, datasets/procesado/) se construyen a
# partir de esta variable, así que basta con abrir el notebook desde la
# carpeta correcta (D: o la carpeta compartida de OneDrive) para que todo
# apunte al lugar correcto sin tocar código.
abspath = pathlib.Path.cwd()
print(f"Directorio de trabajo: {abspath}")
###==========================

# ============================================================
# Periodo de estudio
# ============================================================
# 1_DataExtraction.ipynb descarga automáticamente hasta la fecha más reciente disponible. El
# periodo que analiza el documento se fija aquí, para que actualizar los datos crudos no lo cambie
# por accidente. Para extender el estudio basta con mover FECHA_FIN_ESTUDIO.
FECHA_INICIO_ESTUDIO = pd.Timestamp("2010-01-01")
FECHA_FIN_ESTUDIO = pd.Timestamp("2026-04-30")

# Crea datasets/procesado/ si todavía no existe — ahí se guardan todos los
# CSV intermedios que produce cada bloque de este notebook.
if not os.path.exists(os.path.join(abspath, "datasets", "procesado")):
    os.makedirs(os.path.join(abspath, "datasets", "procesado"))

Directorio de trabajo: C:\Users\Windows 10 Pro\OneDrive\Trabajo Integrador\Trabajo_Integrador_MAIN_PB


## Aportes hídricos

In [2]:
# ============================================================
# Aportes hídricos (AporEner — "Sistema", una sola columna Value diaria)
# ============================================================
aportes = pd.read_csv(os.path.join(abspath,"datasets","AporEner.csv"))
aportes = aportes[['Date','Value']]
aportes = aportes.rename(columns = {'Date': 'fecha',
                                    'Value': 'aportes_energia'}).sort_values(by = 'fecha', ascending = True)

aportes['fecha'] = pd.to_datetime(aportes['fecha'])
aportes = aportes.reset_index(drop=True)
# Guarda la serie procesada; se usa más abajo en la Unión del dataset.
aportes.to_csv(os.path.join(abspath,"datasets", "procesado","aportes.csv"), index=False)
aportes.head(3)

,fecha,aportes_energia
0,2010-01-01,45616100.0
1,2010-01-02,43097100.0
2,2010-01-03,43400400.0


## Compras en bolsa

In [3]:
# ============================================================
# Compras en bolsa (CompBolsNaciEner — "Sistema", 24 columnas horarias)
# ============================================================
# Define la lista de columnas horarias Values_Hour01..24 que traen las
# métricas "Sistema"/"Recurso" descargadas con desagregación horaria (la
# API de XM reporta 24 valores por día). Se reutiliza en las celdas
# siguientes (Demanda, Disponibilidad, Generación, etc.).
hours = [f"Values_Hour{str(i).zfill(2)}" for i in range(1,25)]
compras_bolsa = pd.read_csv(os.path.join(abspath,"datasets","CompBolsNaciEner.csv"))

# Suma las 24 horas para obtener el total diario de energía comprada en bolsa.
compras_bolsa['compras_bolsa'] = compras_bolsa[hours].sum(axis=1)
compras_bolsa = compras_bolsa[ ['Date'] + ['compras_bolsa']]

compras_bolsa = compras_bolsa.rename(columns = {'Date': 'fecha'}).sort_values(by = 'fecha', ascending = True)
compras_bolsa['fecha'] = pd.to_datetime(compras_bolsa['fecha'])
compras_bolsa = compras_bolsa.reset_index(drop=True)
# Nota: compras_bolsa no se incluye en el merge final (ver celda de Unión
# dataset, línea comentada) — queda calculada y guardada, pero fuera del
# dataset consolidado que consumen los modelos.
compras_bolsa.to_csv(os.path.join(abspath,"datasets", "procesado","compras_bolsa.csv"), index=False)
compras_bolsa.head()

,fecha,compras_bolsa
0,2010-01-01,44877553.69
1,2010-01-02,43958164.96
2,2010-01-03,53146284.88
3,2010-01-04,46780069.04
4,2010-01-05,47311962.44


## Compras en contratos

In [4]:
# ============================================================
# Compras en contratos (CompContEner — "Sistema", 24 columnas horarias)
# ============================================================
# Mismo tratamiento que "Compras en bolsa": suma las 24 horas a un total
# diario. Tampoco se incluye en el merge final (ver Unión dataset).
compras_contrato = pd.read_csv(os.path.join(abspath,"datasets","CompContEner.csv"))
compras_contrato['compras_contrato'] = compras_contrato[hours].sum(axis=1)
compras_contrato = compras_contrato[ ['Date'] + ['compras_contrato']]

compras_contrato = compras_contrato.rename(columns = {'Date': 'fecha'}).sort_values(by = 'fecha', ascending = True)
compras_contrato['fecha'] = pd.to_datetime(compras_contrato['fecha'])
compras_contrato.to_csv(os.path.join(abspath,"datasets", "procesado","compras_contrato.csv"), index=False)

## Demanda

In [5]:
# ============================================================
# Demanda (DemaReal — "Sistema", 24 columnas horarias)
# ============================================================
demanda = pd.read_csv(os.path.join(abspath,"datasets","DemaReal.csv"))
demanda['demanda'] = demanda[hours].sum(axis=1)
# demanda = demanda[ ['Date'] + ['demanda']]
demanda = demanda[['Date','demanda']]


demanda = demanda.rename(columns = {'Date': 'fecha',
                                    'demanda': 'demanda'}).sort_values(by = 'fecha', ascending = True)
demanda['fecha'] = pd.to_datetime(demanda['fecha'])
demanda = demanda.reset_index(drop=True)
# Esta sí se incluye en el merge final (ver Unión dataset) — es una de las
# variables base del dataset consolidado.
demanda.to_csv(os.path.join(abspath,"datasets", "procesado","demanda.csv"), index=False)
demanda
# demanda.to_csv('demanda_final.csv', index=False)

,fecha,demanda
0,2010-01-01,1.133616e+08
1,2010-01-02,1.254718e+08
2,2010-01-03,1.211452e+08
3,2010-01-04,1.439317e+08
4,2010-01-05,1.498691e+08
...,...,...
5959,2026-04-26,2.168491e+08
5960,2026-04-27,2.433759e+08
5961,2026-04-28,2.490304e+08
5962,2026-04-29,2.482275e+08


## Listado de Recursos de Generación

In [6]:
# ============================================================
# Listado de Recursos de Generación (catálogo de plantas)
# ============================================================
# No es una serie de tiempo — es el catálogo de plantas (código, tipo de
# generación, combustible, estado), usado más abajo (Disponibilidad
# declarada y Generación) para clasificar cada planta como HIDRAULICA/
# TERMICA/NO TERMICA y así poder desagregar esas variables por tipo.
recursos = pd.read_csv(os.path.join(abspath,"datasets","listado_recursos.csv"))

recursos = recursos[['Values_Code', 'Values_Type', 'Values_EnerSource', 'Values_State']]
recursos = recursos.rename(columns = {'Values_Code': 'codigo_recurso',
                                      'Values_Type': 'tipo_generacion',
                                      'Values_EnerSource':'combustible',
                                      'Values_State':'estado'})
# recursos['fecha'] = pd.to_datetime(recursos['fecha'])
# recursos = recursos.reset_index(drop=True)
recursos.to_csv(os.path.join(abspath,"datasets", "procesado","listado_recursos.csv"), index=False)
print(recursos.columns)
recursos.head(3)

Index(['codigo_recurso', 'tipo_generacion', 'combustible', 'estado'], dtype='object')


,codigo_recurso,tipo_generacion,combustible,estado
0,2QBW,HIDRAULICA,AGUA,OPERACION
1,2QEK,HIDRAULICA,AGUA,OPERACION
2,2QRL,HIDRAULICA,AGUA,OPERACION


## Disponibilidad declarada

> **Corrección del 18/10/2026 — agregación de la disponibilidad declarada.** > > Esta celda agregaba la disponibilidad con **media** en los dos pasos (sobre las 24 horas > y sobre las plantas), mientras la generación usa **suma** en ambos.

In [7]:
# ============================================================
# Disponibilidad declarada (DispoDeclarada — "Recurso", por planta)
# ============================================================
# Clasifica cada planta en 3 categorías simplificadas a partir del
# tipo_generacion del catálogo de recursos: HIDRAULICA, TERMICA, o
# NO TERMICA (todo lo que no es hidráulica ni térmica: solar, eólica,
# biomasa, etc.). Esta función se reutiliza también en la celda de
# Generación, más abajo.
def tipo_gen(row):
    if row['tipo_generacion'] == 'TERMICA':
        return 'TERMICA'

    elif row['tipo_generacion'] == 'HIDRAULICA':
        return 'HIDRAULICA'
    elif row['tipo_generacion'] != 'TERMICA':
        return 'NO TERMICA'

# Crear la nueva columna basada en la función condicion
# disponibilidad_declarada = pd.read_csv('disponibilidad_declarada.csv')
disponibilidad_declarada = pd.read_csv(os.path.join(abspath,"datasets","DispoDeclarada.csv"))
# Suma horaria -> energía diaria por planta, igual que la generación.
# La suma, y no la media, es lo que deja la variable comparable con generacion y con
# demanda, que tambien se suman sobre las 24 horas; promediarla la dejaria en kWh por hora.
disponibilidad_declarada['disponibilidad_declarada'] = disponibilidad_declarada[hours].sum(axis=1)
# Cruce con el catálogo de recursos para poder clasificar cada código de
# planta (Values_code) por tipo de generación.
disponibilidad_declarada = pd.merge(disponibilidad_declarada, recursos, how = 'left', left_on = 'Values_code', right_on = 'codigo_recurso' )
# Total diario del SISTEMA: suma sobre todas las plantas, antes de desagregar por tipo.
# Se suma, no se promedia: el promedio daria el tamaño medio de la planta que reporta y no
# la disponibilidad del sistema. El numero de plantas pasa de 55 en 2010 a 495 en 2026, de
# modo que la media cae un 82 % mientras la demanda crece un 53 %.
disponibilidad_declarada_total = disponibilidad_declarada.groupby(['Date'])['disponibilidad_declarada'].sum().reset_index()


disponibilidad_declarada['tipo_generacion_aux'] = disponibilidad_declarada.apply(lambda row: tipo_gen(row), axis=1)

# Suma diaria por tipo de generación (HIDRAULICA/TERMICA/NO TERMICA). Con suma, las tres
# componentes sí reconstruyen el total; con media no lo hacían, porque el total era la
# media sobre TODAS las plantas y cada componente la media sobre las de su tecnología.
disponibilidad_declarada = disponibilidad_declarada.groupby(['Date','tipo_generacion_aux'])['disponibilidad_declarada'].sum().reset_index()
# disponibilidad_declarada = disponibilidad_declarada[disponibilidad_declarada['tipo_generacion_aux'].isin(['HIDRAULICA', 'TERMICA'])]
# Pivotea de formato largo (una fila por Date+tipo) a ancho (una columna por tipo).
disponibilidad_declarada = pd.pivot_table(disponibilidad_declarada, index = 'Date', columns = 'tipo_generacion_aux', values = 'disponibilidad_declarada').reset_index()
disponibilidad_declarada = disponibilidad_declarada.rename(columns = {
                                                                      'Date': 'fecha',
                                                                      'HIDRAULICA': 'disp_dec_hidraulica',
                                                                      'TERMICA': 'disp_dec_termica',
                                                                      'NO TERMICA': 'disp_dec_no_termica'})

disponibilidad_declarada_total = disponibilidad_declarada_total.rename(columns = {
                                                                      'Date': 'fecha'})
disponibilidad_declarada['fecha'] = pd.to_datetime(disponibilidad_declarada['fecha'])
disponibilidad_declarada_total['fecha'] = pd.to_datetime(disponibilidad_declarada_total['fecha'])
# Une el desagregado por tipo con el total general (columna disponibilidad_declarada).
disponibilidad_declarada = pd.merge(disponibilidad_declarada, disponibilidad_declarada_total, how = 'left', on = 'fecha')
disponibilidad_declarada.to_csv(os.path.join(abspath,"datasets", "procesado","disponibilidad_declarada.csv"), index=False)
disponibilidad_declarada.head(3)

,fecha,disp_dec_hidraulica,disp_dec_no_termica,disp_dec_termica,disponibilidad_declarada
0,2010-01-01,188034000.0,4778000.0,89259000.0,282071000.0
1,2010-01-02,186794000.0,4778000.0,89274000.0,280846000.0
2,2010-01-03,188274000.0,4778000.0,87810000.0,280862000.0


## Generación

In [8]:
# ============================================================
# Generación (Gene — "Recurso", por planta)
# ============================================================
# Mismo patrón que "Disponibilidad declarada": clasifica por tipo de
# generación (tipo_gen), desagrega HIDRAULICA/TERMICA/NO TERMICA, y agrega
# el total general.
# Crear la nueva columna basada en la función condicion
# generacion = pd.read_csv('generacion.csv')
generacion = pd.read_csv(os.path.join(abspath,"datasets","Gene.csv"))
# Suma horaria (no promedio, a diferencia de disponibilidad) -> energía diaria generada.
generacion['generacion'] = generacion[hours].sum(axis=1)
generacion = pd.merge(generacion, recursos, how = 'left', left_on = 'Values_code', right_on = 'codigo_recurso' )
# Total diario (todas las plantas), suma en vez de promedio.
generacion_total = generacion.groupby(['Date'])['generacion'].sum().reset_index()


generacion['tipo_generacion_aux'] = generacion.apply(lambda row: tipo_gen(row), axis=1)

# Suma diaria por tipo de generación.
generacion = generacion.groupby(['Date','tipo_generacion_aux'])['generacion'].sum().reset_index()
# generacion = generacion[generacion['tipo_generacion_aux'].isin(['HIDRAULICA', 'TERMICA'])]
generacion = pd.pivot_table(generacion, index = 'Date', columns = 'tipo_generacion_aux', values = 'generacion').reset_index()
generacion = generacion.rename(columns = {
                                                                      'Date': 'fecha',
                                                                      'HIDRAULICA': 'gen_hidraulica',
                                                                      'TERMICA': 'gen_termica',
                                                                      'NO TERMICA': 'gen_no_termica'})

generacion_total = generacion_total.rename(columns = {
                                                                      'Date': 'fecha'})
generacion['fecha'] = pd.to_datetime(generacion['fecha'])
generacion_total['fecha'] = pd.to_datetime(generacion_total['fecha'])
generacion = pd.merge(generacion, generacion_total, how = 'left', on = 'fecha')
generacion.to_csv(os.path.join(abspath,"datasets", "procesado","generacion.csv"), index=False)
generacion.head(3)

,fecha,gen_hidraulica,gen_no_termica,gen_termica,generacion
0,2010-01-01,35796804.67,6020226.83,73602052.96,1.154191e+08
1,2010-01-02,46595646.99,6050741.63,75034806.16,1.276812e+08
2,2010-01-03,42578875.20,5702074.83,75356828.41,1.236378e+08


## Precio de bolsa

In [9]:
# ============================================================
# Precio de bolsa (PrecBolsNaci — "Sistema") — VARIABLE OBJETIVO
# ============================================================
# disponibilidad_real = pd.read_csv('disponibilidad_real.csv')
precio_bolsa = pd.read_csv(os.path.join(abspath,"datasets","PrecBolsNaci.csv"))
# Promedio de las 24 horas -> precio diario.
precio_bolsa['precio_bolsa'] = precio_bolsa[hours].mean(axis=1)
# groupby+mean adicional (redundante con el promedio horario de arriba, pero
# inofensivo: no hay más de una fila por Date en el crudo).
precio_bolsa = precio_bolsa.groupby('Date')['precio_bolsa'].mean().reset_index()
precio_bolsa = precio_bolsa[ ['Date'] + ['precio_bolsa']]
precio_bolsa = precio_bolsa.rename(columns = {'Date': 'fecha'}).sort_values(by = 'fecha', ascending = True).reset_index(drop = True)
precio_bolsa['fecha'] = pd.to_datetime(precio_bolsa['fecha'])
precio_bolsa.to_csv(os.path.join(abspath,"datasets", "procesado","precio_bolsa.csv"), index=False)
precio_bolsa.head()

,fecha,precio_bolsa
0,2010-01-01,120.501228
1,2010-01-02,118.495340
2,2010-01-03,121.160145
3,2010-01-04,123.962937
4,2010-01-05,120.666437


## Precio de oferta

In [10]:
# ============================================================
# Precio de oferta (PrecOferDesp — "Recurso", por planta)
# ============================================================
# ⚠️ Nota metodológica (ver sección 10 del documento de análisis de
# estructura): precio_oferta tiene correlación alta (0.765) con
# precio_bolsa, porque el precio de bolsa se calcula justamente a partir
# del precio de oferta de la planta marginal despachada. No es un problema
# de este notebook, pero conviene tenerlo presente al interpretar su peso
# en los modelos.
# disponibilidad_real = pd.read_csv('disponibilidad_real.csv')
precio_oferta = pd.read_csv(os.path.join(abspath,"datasets","PrecOferDesp.csv"))
# Promedio horario por planta, luego promedio simple entre todas las plantas
# (no ponderado por generación ni filtrado por tipo de combustible — se
# evaluó desagregar por combustible y se descartó por redundancia, ver
# sección 10.1 del documento de análisis).
precio_oferta['precio_oferta'] = precio_oferta[hours].mean(axis=1)
precio_oferta = precio_oferta.groupby('Date')['precio_oferta'].mean().reset_index()
precio_oferta = precio_oferta[ ['Date'] + ['precio_oferta']]
precio_oferta = precio_oferta.rename(columns = {'Date': 'fecha'}).sort_values(by = 'fecha', ascending = True).reset_index(drop = True)
precio_oferta['fecha'] = pd.to_datetime(precio_oferta['fecha'])
precio_oferta.to_csv(os.path.join(abspath,"datasets", "procesado","precio_oferta.csv"), index=False)
precio_oferta.head()

,fecha,precio_oferta
0,2010-01-01,267.437633
1,2010-01-02,277.153592
2,2010-01-03,313.552755
3,2010-01-04,303.519878
4,2010-01-05,339.600449


## Precio de Escasez

In [11]:
# ============================================================
# Precio de Escasez (CREG, vía API de XM — métrica PrecEsca)
# ============================================================
# Se incluye como proxy del precio de los combustibles: la CREG lo actualiza
# con base en la variación de un índice de precios de combustibles (ver
# sección 10 del documento de análisis de estructura del repo). No es un
# precio de mercado (cambia poco, ~119 valores distintos en 10 años).
#
# ⚠️ ADVERTENCIA SOBRE LA CORRELACIÓN DE ESTA VARIABLE: un cálculo sobre un subperiodo reporta
# de -0.211 con precio_bolsa; al recalcularlo sobre el dataset real y
# completo (2010-2026) dio +0.292. No fue un
# error de cálculo: el signo cambia según la ventana temporal usada (el
# cálculo viejo se hizo sin querer contra la versión corta 2010-2019 de
# precio_bolsa). Esa inestabilidad es en sí misma un hallazgo relevante —
# refuerza que precio_escasez se comporta como una variable administrativa
# de tendencia lenta y no como una señal de mercado genuina, y que su
# correlación con precio_bolsa no debe tomarse como un número fijo y
# confiable. Con la matriz de correlación completa del dataset extendido
# (sección 17), sus únicas correlaciones >0.8 son con el bloque IPC/IPP en
# NIVEL (que la selección de variables descarta) — no es redundante con nada de lo
# que se conserva en el dataset final.
#
# Igual que aportes/vol_util/vertimientos, PrecEsca viene de la API como una
# sola columna "Value" diaria (no desagregada por horas), así que el
# procesamiento es directo, sin necesidad de promediar/sumar columnas Values_HourXX.
precio_escasez = pd.read_csv(os.path.join(abspath,"datasets","PrecEsca.csv"))
precio_escasez = precio_escasez[['Date','Value']]
precio_escasez = precio_escasez.rename(columns = {'Date': 'fecha',
                                    'Value': 'precio_escasez'}).sort_values(by = 'fecha', ascending = True)
precio_escasez['fecha'] = pd.to_datetime(precio_escasez['fecha'])
precio_escasez = precio_escasez.reset_index(drop=True)
precio_escasez.to_csv(os.path.join(abspath,"datasets", "procesado","precio_escasez.csv"), index=False)
precio_escasez

,fecha,precio_escasez
0,2010-01-01,356.70894
1,2010-01-02,355.06286
2,2010-01-03,355.07299
3,2010-01-04,355.07299
4,2010-01-05,354.78317
...,...,...
5959,2026-04-26,763.25043
5960,2026-04-27,763.25043
5961,2026-04-28,763.25043
5962,2026-04-29,763.25043


## Ventas en bolsa nacional

In [12]:
# ============================================================
# Ventas en bolsa nacional (VentBolsNaciEner — "Sistema", 24 columnas horarias)
# ============================================================
# Redefine "hours" (redundante, ya se definió en la celda de Compras en
# bolsa, pero no hace daño). No se incluye en el merge final (ver Unión dataset).
hours = [f"Values_Hour{str(i).zfill(2)}" for i in range(1,25)]
ventas_bolsa = pd.read_csv(os.path.join(abspath,"datasets","VentBolsNaciEner.csv"))

ventas_bolsa['ventas_bolsa'] = ventas_bolsa[hours].sum(axis=1)
ventas_bolsa = ventas_bolsa[ ['Date'] + ['ventas_bolsa']]

ventas_bolsa = ventas_bolsa.rename(columns = {'Date': 'fecha'}).sort_values(by = 'fecha', ascending = True)
ventas_bolsa['fecha'] = pd.to_datetime(ventas_bolsa['fecha'])
ventas_bolsa = ventas_bolsa.reset_index(drop=True)
ventas_bolsa.to_csv(os.path.join(abspath,"datasets", "procesado","ventas_bolsa.csv"), index=False)
ventas_bolsa.head()

,fecha,ventas_bolsa
0,2010-01-01,44877553.69
1,2010-01-02,43958164.96
2,2010-01-03,53146284.87
3,2010-01-04,46759694.44
4,2010-01-05,47302938.66


## Ventas en contratos

In [13]:
# ============================================================
# Ventas en contratos (VentContEner — "Sistema", 24 columnas horarias)
# ============================================================
# Mismo tratamiento que "Ventas en bolsa nacional". Tampoco se incluye en
# el merge final (ver Unión dataset).
hours = [f"Values_Hour{str(i).zfill(2)}" for i in range(1,25)]
ventas_contrato = pd.read_csv(os.path.join(abspath,"datasets","VentContEner.csv"))

ventas_contrato['ventas_contrato'] = ventas_contrato[hours].sum(axis=1)
ventas_contrato = ventas_contrato[ ['Date'] + ['ventas_contrato']]

ventas_contrato = ventas_contrato.rename(columns = {'Date': 'fecha'}).sort_values(by = 'fecha', ascending = True)
ventas_contrato['fecha'] = pd.to_datetime(ventas_contrato['fecha'])
ventas_contrato = ventas_contrato.reset_index(drop=True)
ventas_contrato.to_csv(os.path.join(abspath,"datasets", "procesado","ventas_contrato.csv"), index=False)
ventas_contrato.head()

,fecha,ventas_contrato
0,2010-01-01,1.395132e+08
1,2010-01-02,1.490199e+08
2,2010-01-03,1.427063e+08
3,2010-01-04,1.623375e+08
4,2010-01-05,1.644152e+08


## Volumen útil

In [14]:
# ============================================================
# Volumen útil (VoluUtilDiarEner — "Sistema", una sola columna Value diaria)
# ============================================================
vol_util = pd.read_csv(os.path.join(abspath,"datasets","VoluUtilDiarEner.csv"))
# vol_util['vol_util'] = vol_util[hours].sum(axis=1)
# vol_util = vol_util[ ['Date'] + ['vol_util']]
vol_util = vol_util[['Date','Value']]


vol_util = vol_util.rename(columns = {'Date': 'fecha',
                                    'Value': 'vol_util'}).sort_values(by = 'fecha', ascending = True)
vol_util['fecha'] = pd.to_datetime(vol_util['fecha'])
vol_util = vol_util.reset_index(drop=True)
# Se incluye en el merge final (ver Unión dataset) — volumen útil de embalses.
vol_util.to_csv(os.path.join(abspath,"datasets", "procesado","vol_util.csv"), index=False)
vol_util

,fecha,vol_util
0,2010-01-01,1.000661e+10
1,2010-01-02,9.995147e+09
2,2010-01-03,9.992677e+09
3,2010-01-04,9.965341e+09
4,2010-01-05,9.926760e+09
...,...,...
5959,2026-04-26,1.074502e+10
5960,2026-04-27,1.078151e+10
5961,2026-04-28,1.080782e+10
5962,2026-04-29,1.085773e+10


## Vertimientos

In [15]:
# ============================================================
# Vertimientos (VertEner — "Sistema", una sola columna Value diaria)
# ============================================================
# ⚠️ Esta serie tiene muchos días sin fila (no todos los días hay
# vertimientos que reportar) — los nulos que deja tras el merge final se
# imputan a 0 en el notebook de análisis exploratorio, asumiendo que un día sin dato significa
# "no hubo vertimiento".
vertimientos = pd.read_csv(os.path.join(abspath,"datasets","VertEner.csv"))
# vertimientos['vertimientos'] = vertimientos[hours].sum(axis=1)
# vertimientos = vertimientos[ ['Date'] + ['vertimientos']]
vertimientos = vertimientos[['Date','Value']]


vertimientos = vertimientos.rename(columns = {'Date': 'fecha',
                                    'Value': 'vertimientos'}).sort_values(by = 'fecha', ascending = True)
vertimientos['fecha'] = pd.to_datetime(vertimientos['fecha'])
vertimientos = vertimientos.reset_index(drop=True)
vertimientos.to_csv(os.path.join(abspath,"datasets", "procesado","vertimientos.csv"), index=False)
vertimientos

,fecha,vertimientos
0,2010-02-15,4.259369e+03
1,2010-02-22,6.366273e+05
2,2010-02-23,7.226934e+05
3,2010-02-24,8.421727e+05
4,2010-02-25,8.327154e+05
...,...,...
3646,2026-04-26,8.814859e+06
3647,2026-04-27,8.349438e+06
3648,2026-04-28,9.220207e+06
3649,2026-04-29,1.333685e+07


## Indice ONI

In [16]:
# ============================================================
# Índice ONI en dos versiones
# ============================================================
# Cada valor del ONI es la media de tres meses de la anomalía de temperatura del
# Pacífico (región Niño 3.4), y NOAA lo asigna al mes central: el valor de junio
# promedia mayo, junio y julio. NOAA actualiza la tabla antes del día 5 del mes
# siguiente al cierre de la temporada
# (https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso/oni/v6/), de modo
# que el valor asignado al mes m solo se conoce desde el día 6 del mes m+2.
#
#   indice_ONI            Versión centrada: cada día toma el valor de la temporada
#                         centrada en su mes. Describe qué fase hubo cada día y es la
#                         que usa el análisis exploratorio (3_EDA). No debe entrar a los
#                         modelos: contiene información de hasta dos meses después.
#   indice_ONI_publicado  Versión publicada: cada valor rige desde el día en que NOAA lo
#                         publicó, de modo que cada día ve solo lo que ya se conocía. Es
#                         la que deben usar la selección de variables, los modelos y la
#                         evaluación por fase.
#
# Limitación: NOAA revisa los valores hasta dos meses después de publicarlos y
# actualiza su periodo base cada cinco años, así que el archivo actual trae valores
# revisados y no necesariamente el número exacto que se publicó en su momento.
ONI = pd.read_excel(os.path.join(abspath,"datasets","Indice_ONI.xlsx"))
oni_mensual = pd.melt(ONI, id_vars=['Year'], var_name='Month', value_name='indice_ONI')
# FIX 2026-09-01: "Month" ahora puede venir como entero (1-12) en vez de string,
# porque el Paso 6 automatizado de 1_DataExtraction.ipynb genera las columnas de
# mes como int (list(range(1,13))), mientras que la versión mantenida a mano del
# .xlsx las traía como texto. Se castea explícitamente a str para que la
# concatenación con Year (también string) funcione sin importar el tipo de origen.
oni_mensual['fecha'] = pd.to_datetime(oni_mensual['Year'].astype(str) + '-' + oni_mensual['Month'].astype(str), format='%Y-%m')
oni_mensual = oni_mensual.drop(['Year', 'Month'], axis=1).sort_values(by='fecha').set_index('fecha')

# (a) Versión centrada: la misma construcción que se usaba hasta ahora.
oni_centrado = oni_mensual.resample('D').asfreq().ffill()

# (b) Versión publicada: el valor del mes m rige desde el día 6 del mes m+2.
oni_publicado = oni_mensual['indice_ONI'].dropna()
oni_publicado.index = oni_publicado.index + pd.DateOffset(months=2) + pd.Timedelta(days=5)
oni_publicado = oni_publicado.rename('indice_ONI_publicado').resample('D').asfreq().ffill()

# Se unen y se recortan al periodo de estudio: el xlsx incluye 2009 solo para poder
# construir la versión publicada de enero y febrero de 2010.
indice_ONI = oni_centrado.join(oni_publicado, how='outer')
indice_ONI = indice_ONI[indice_ONI.index >= '2010-01-01'].reset_index()
indice_ONI.to_csv(os.path.join(abspath,"datasets", "procesado","Indice_ONI.csv"), index=False)
indice_ONI.head()

,fecha,indice_ONI,indice_ONI_publicado
0,2010-01-01,1.47,0.92
1,2010-01-02,1.47,0.92
2,2010-01-03,1.47,0.92
3,2010-01-04,1.47,0.92
4,2010-01-05,1.47,0.92


## IPC — Índice de Precios al Consumidor

In [17]:
# ============================================================
# IPC: Índice de Precios al Consumidor (DANE)
# ============================================================
# Se incluye como proxy de inflación general. Se calculan dos versiones,
# ambas documentadas explícitamente (ver sección 9.4 del análisis de estructura):
#   - ipc_nivel: el valor del índice tal cual.
#   - ipc_variacion_interanual: variación % contra el mismo mes del año
#     anterior — la forma estándar en que DANE/BanRep reportan la inflación.
#     El nivel del índice tiene tendencia determinística fuerte (como el
#     precio de bolsa), así que la variación interanual ayuda a evitar
#     correlación espuria; la decisión de cuál conservar corresponde a la selección de variables.
# El IPC es mensual en origen; se expande a diario por forward-fill, igual
# que el índice ONI, para poder unirlo al dataset diario sin perder
# granularidad (ver discusión de por qué diario y no mensual/horario).
#
# Se lee el anexo del DANE que descarga automáticamente
# 1_DataExtraction.ipynb (Paso 7), en lugar del archivo del Banco de la República
# que se bajaba a mano. Es la misma serie: coinciden en todos los meses desde 2009.
# El anexo viene como matriz, con los meses en filas y los años en columnas.
# Cada mes se fecha en su ÚLTIMO día, como venía en el archivo del Banco de la
# República, para reproducir exactamente el dataset anterior: al expandir a diario,
# el valor de un mes rige desde su último día (el DANE lo publica en los primeros
# días del mes siguiente).
MESES_ES_IPC = {
    'Enero': 1, 'Febrero': 2, 'Marzo': 3, 'Abril': 4, 'Mayo': 5, 'Junio': 6,
    'Julio': 7, 'Agosto': 8, 'Septiembre': 9, 'Octubre': 10, 'Noviembre': 11,
    'Diciembre': 12,
}
ipc_raw = pd.read_excel(os.path.join(abspath, "datasets", "IPC_DANE.xlsx"), header=None)
fila_mes = ipc_raw.index[ipc_raw.iloc[:, 0].astype(str).str.strip() == 'Mes'][0]
anios = list(ipc_raw.iloc[fila_mes, 1:])
bloque = ipc_raw.iloc[fila_mes + 1: fila_mes + 13].copy()
bloque.columns = ['mes_txt'] + anios
ipc = bloque.melt(id_vars='mes_txt', var_name='anio', value_name='ipc_nivel')
ipc['anio'] = pd.to_numeric(ipc['anio'], errors='coerce')
ipc['ipc_nivel'] = pd.to_numeric(ipc['ipc_nivel'], errors='coerce')
ipc = ipc.dropna(subset=['anio', 'ipc_nivel'])
ipc['fecha'] = pd.to_datetime(dict(year=ipc['anio'].astype(int),
                                   month=ipc['mes_txt'].astype(str).str.strip().map(MESES_ES_IPC),
                                   day=1)) + pd.offsets.MonthEnd(0)
ipc = ipc[['fecha', 'ipc_nivel']].sort_values('fecha').reset_index(drop=True)

# Variación interanual (%): cada fila es un mes, así que comparar contra la
# fila 12 posiciones atrás equivale a comparar contra el mismo mes del año anterior.
ipc['ipc_variacion_interanual'] = ipc['ipc_nivel'].pct_change(periods=12) * 100

# Expandir de mensual a diario por forward-fill (mismo tratamiento que el ONI).
ipc = ipc.set_index('fecha')
ipc = ipc.resample('D').asfreq().ffill()
ipc = ipc.reset_index()

ipc.to_csv(os.path.join(abspath, "datasets", "procesado", "ipc.csv"), index=False)
ipc.tail()

,fecha,ipc_nivel,ipc_variacion_interanual
8609,2026-08-27,159.79,6.024816
8610,2026-08-28,159.79,6.024816
8611,2026-08-29,159.79,6.024816
8612,2026-08-30,159.79,6.024816
8613,2026-08-31,160.42,6.245447


## IPP — Índice de Precios del Productor

In [18]:
# ============================================================
# IPP: Índice de Precios del Productor (DANE)
# ============================================================
# Es un anexo de reporte (no una tabla tabular limpia): título/encabezados en
# las primeras filas, el año solo aparece en la primera fila de cada año, el
# mes viene en español (a veces con sufijo "(pr*)" de cifra provisional), y
# hay notas al pie al final. Se seleccionan 3 series, justificadas en la
# sección 9.3 del documento de análisis de estructura del repo:
#   - Oferta Interna - Minería  -> costo de combustibles fósiles domésticos
#                                  (carbón, gas) para generación térmica.
#   - Procedencias - Importados -> costo de combustibles importados y
#                                  traslado de tasa de cambio.
#   - Oferta Interna (general) -> control macroeconómico amplio.
# Se descartan Agricultura/ganadería/pesca y Exportados por falta de
# mecanismo causal claro con el precio de bolsa. Se usa "Oferta Interna" en
# vez de "Producción Nacional" porque representa mejor los costos que
# enfrenta el sistema eléctrico colombiano (mercado doméstico).
# Al igual que el IPC y el ONI, se calcula nivel y variación interanual (%),
# y se expande de mensual a diario por forward-fill.

MESES_ES = {
    'Enero': 1, 'Febrero': 2, 'Marzo': 3, 'Abril': 4, 'Mayo': 5, 'Junio': 6,
    'Julio': 7, 'Agosto': 8, 'Septiembre': 9, 'Octubre': 10, 'Noviembre': 11,
    'Diciembre': 12,
}

# header=None porque el archivo no tiene un encabezado tabular estándar;
# skiprows=5 descarta título/subtítulos/encabezados fusionados (filas 1-5).
ipp_raw = pd.read_excel(
    os.path.join(abspath, "datasets", "IPP_DANE.xlsx"),
    sheet_name="IPP Histórico",
    header=None,
    skiprows=5,
)

# Columnas por posición, según la estructura real del archivo:
# 0: Año (solo en la primera fila de cada año) | 1: Mes (texto en español)
# 2: Producción Nacional (general)   | 3: Prod. Nac. - Agricultura/ganadería/pesca
# 4: Prod. Nac. - Minería            | 5: Prod. Nac. - Industria
# 6: Oferta Interna (general)        | 7: Oferta Interna - Agricultura/ganadería/pesca
# 8: Oferta Interna - Minería        | 9: Oferta Interna - Industria
# 10: Procedencias - Consumo interno | 11: Procedencias - Importados
# 12: Procedencias - Exportados
ipp_raw = ipp_raw.iloc[:, :13]
ipp_raw.columns = [
    'anio', 'mes_txt', 'prod_nac_general', 'prod_nac_agro', 'prod_nac_mineria',
    'prod_nac_industria', 'oferta_interna_general', 'oferta_interna_agro',
    'oferta_interna_mineria', 'oferta_interna_industria', 'proc_consumo_interno',
    'proc_importados', 'proc_exportados',
]

# ⚠️ El año viene como string vacío '' en algunas filas en vez de NaN real
# (ej. la fila de abril-2026) — hay que normalizarlo a NaN antes del
# forward-fill, o si no el forward-fill no rellena esas filas y se pierde el año.
ipp_raw['anio'] = ipp_raw['anio'].replace('', pd.NA)
ipp_raw['anio'] = ipp_raw['anio'].ffill()
ipp_raw['anio'] = pd.to_numeric(ipp_raw['anio'], errors='coerce')

# Se queda solo con filas de datos reales (año numérico y mes no nulo);
# esto descarta automáticamente las filas de notas al pie del final, que no
# tienen un año numérico válido en la columna A.
ipp = ipp_raw.dropna(subset=['anio', 'mes_txt']).copy()

# Limpia el sufijo de cifra provisional, ej. "Junio (pr*)" -> "Junio".
ipp['mes_txt'] = ipp['mes_txt'].astype(str).str.replace(r'\s*\(pr\*?\)', '', regex=True).str.strip()
ipp['mes_num'] = ipp['mes_txt'].map(MESES_ES)
ipp = ipp.dropna(subset=['mes_num'])  # por si queda alguna fila de nota no descartada antes

ipp['fecha'] = pd.to_datetime(
    ipp['anio'].astype(int).astype(str) + '-' + ipp['mes_num'].astype(int).astype(str) + '-01'
)

# Selección de las 3 series justificadas económicamente (sección 9.3).
ipp = ipp[['fecha', 'oferta_interna_mineria', 'proc_importados', 'oferta_interna_general']]
ipp = ipp.rename(columns={
    'oferta_interna_mineria': 'ipp_mineria_nivel',
    'proc_importados': 'ipp_importados_nivel',
    'oferta_interna_general': 'ipp_general_nivel',
})
ipp = ipp.sort_values('fecha').reset_index(drop=True)

# Variación interanual (%) para cada una de las 3 series.
for col in ['ipp_mineria_nivel', 'ipp_importados_nivel', 'ipp_general_nivel']:
    var_col = col.replace('_nivel', '_variacion_interanual')
    ipp[var_col] = ipp[col].pct_change(periods=12) * 100

# Expandir de mensual a diario por forward-fill (mismo tratamiento que ONI e IPC).
ipp = ipp.set_index('fecha')
ipp = ipp.resample('D').asfreq().ffill()
ipp = ipp.reset_index()

ipp.to_csv(os.path.join(abspath, "datasets", "procesado", "ipp.csv"), index=False)
ipp.tail()

,fecha,ipp_mineria_nivel,ipp_importados_nivel,ipp_general_nivel,ipp_mineria_variacion_interanual,ipp_importados_variacion_interanual,ipp_general_variacion_interanual
9919,2026-07-28,173.60,146.84,186.15,-4.141358,-6.874683,1.311636
9920,2026-07-29,173.60,146.84,186.15,-4.141358,-6.874683,1.311636
9921,2026-07-30,173.60,146.84,186.15,-4.141358,-6.874683,1.311636
9922,2026-07-31,173.60,146.84,186.15,-4.141358,-6.874683,1.311636
9923,2026-08-01,170.24,143.69,185.33,-8.280804,-8.843494,0.384574


## Variables de calendario

In [19]:
# ============================================================
# Variables de calendario: día de semana, día del mes, mes, y tipo de día
# ============================================================
# Motivación (Rahman y Reza, 2025; Giraldo Z. y Ramírez L., 2023):
# varios trabajos de la revisión bibliográfica incluyen variables de
# calendario como predictoras del precio spot, porque la demanda eléctrica
# (y por tanto el precio) tiene un patrón claro asociado al tipo de día:
#   - Rahman (2025): weekday/weekend binario + indicador "midweek", sin
#     distinguir festivos.
#   - Kizildag (2024): variable categórica laboral/fin de semana/festivo.
#   - GiraldoRamirez (2023) — el más relevante, es sobre el MEM colombiano:
#     etiqueta "day_cls" vía K-means sobre demanda des-tendenciada, con 3
#     categorías (ordinario L-V, sábado, festivo incluyendo domingo).
# Nuestra variable "tipo_dia" es un refinamiento del enfoque de
# GiraldoRamirez: en vez de agrupar domingo y festivo en una sola
# categoría, los separamos (4 categorías en total), porque el patrón de
# demanda de un domingo corriente puede diferir del de un festivo entre
# semana (ej. puentes largos, festivos de Ley Emiliani).
#
# Definición adoptada en este trabajo:
#   0 = Ordinario (lunes a viernes, no festivo)
#   1 = Sábado
#   2 = Domingo
#   3 = Festivo
#
# Regla de prioridad para el caso ambiguo "festivo que cae en domingo"
# (ej. 20-jul-2025, 8-dic-2024, ambas fechas fijas — no se mueven por Ley
# Emiliani): FESTIVO TIENE PRIORIDAD sobre domingo (decisión adoptada en
# este trabajo). Un festivo civil se trata igual sin
# importar el día de la semana en que caiga.
#
# Fuente del calendario de festivos: librería `holidays` (paquete de
# Python, ver import en la celda de configuración inicial), objeto
# holidays.Colombia(). Se verificó contra el calendario oficial 2024-2025
# antes de usarla: aplica correctamente la Ley Emiliani (traslada al lunes
# siguiente los festivos "trasladables": Reyes Magos, San José, San Pedro y
# San Pablo, La Asunción, Día de la Raza, Todos los Santos, Independencia
# de Cartagena, Ascensión, Corpus Christi, Sagrado Corazón) y deja fijos
# los que no se trasladan (Año Nuevo, Día del Trabajo, Independencia,
# Batalla de Boyacá, Inmaculada Concepción, Navidad, Jueves y Viernes
# Santo). Promedio verificado: ~18 festivos/año, igual al calendario civil
# colombiano oficial.
festivos_co = holidays.Colombia(years=range(2010, 2027))

# --- Variables simples directas desde la columna fecha ---
# dia_semana: convención estándar de pandas/Python — Monday=0 ... Sunday=6.
# dia_mes: 1-31. mes: 1-12. Se calculan sobre el rango completo del
# dataset (usa las fechas de precio_bolsa, ya procesado más arriba, como
# referencia — mismo rango que el resto del dataset consolidado).
calendario = precio_bolsa[['fecha']].copy()
calendario['dia_semana'] = calendario['fecha'].dt.dayofweek
calendario['dia_mes'] = calendario['fecha'].dt.day
calendario['mes'] = calendario['fecha'].dt.month


def clasificar_tipo_dia(row):
    """Clasifica una fecha en 0=Ordinario, 1=Sábado, 2=Domingo, 3=Festivo.

    El festivo tiene prioridad sobre el domingo cuando coinciden (regla
    adoptada en este trabajo) — se evalúa primero.
    """
    es_festivo = row['fecha'].date() in festivos_co
    if es_festivo:
        return 3
    elif row['dia_semana'] == 6:  # domingo
        return 2
    elif row['dia_semana'] == 5:  # sábado
        return 1
    else:
        return 0  # lunes a viernes, no festivo


calendario['tipo_dia'] = calendario.apply(clasificar_tipo_dia, axis=1)

# --- Validación ---
# Se verificó: (1) el total de filas por categoría suma el total de días
# del dataset; (2) los festivos por año rondan 18 (algunos años dan 17
# porque dos festivos distintos caen el mismo día calendario, ej. Sagrado
# Corazón y San Pedro y San Pablo en 2025); (3) los casos ambiguos
# festivo+domingo (20-jul-2025, 8-dic-2024) quedan clasificados como
# tipo_dia=3, conforme a la regla de prioridad acordada.
print("Distribución tipo_dia:")
print(calendario['tipo_dia'].value_counts().sort_index())

calendario.to_csv(os.path.join(abspath, "datasets", "procesado", "calendario.csv"), index=False)
calendario.head()

Distribución tipo_dia:
tipo_dia
0    4001
1     839
2     835
3     289
Name: count, dtype: int64


,fecha,dia_semana,dia_mes,mes,tipo_dia
0,2010-01-01,4,1,1,3
1,2010-01-02,5,2,1,1
2,2010-01-03,6,3,1,2
3,2010-01-04,0,4,1,0
4,2010-01-05,1,5,1,0


# Unión dataset

In [20]:
# ============================================================
# Unión del dataset: merge de todas las variables procesadas, por 'fecha'
# ============================================================
# join_type='left' con precio_bolsa como base: todas las variables se unen
# sobre las fechas que sí tienen precio de bolsa (la variable objetivo).
# SEIS SERIES SE CALCULAN Y NO SE INTEGRAN, y las líneas comentadas lo dejan a la vista:
# la demanda del Sistema Interconectado Nacional, la disponibilidad real, y las compras y
# las ventas de energía en bolsa y en contratos. La demanda del SIN reproduce la demanda
# real, de la que difiere en promedio un 1,4 %; las otras cinco no tienen respaldo en los
# estudios de la revisión de literatura ni forman parte del conjunto que declara el trabajo
# de referencia sobre este mercado. Se conservan extraídas y procesadas, fuera del conjunto
# de modelamiento, por si una extensión del trabajo las requiere.
join_type = 'left'
df_final = pd.merge(precio_bolsa, demanda, how = join_type, on = 'fecha')
# df_final = pd.merge(df_final, compras_contrato, how = join_type, on = 'fecha')
# df_final = pd.merge(df_final, compras_bolsa, how = join_type, on = 'fecha')
df_final = pd.merge(df_final, disponibilidad_declarada, how = join_type, on = 'fecha')
# df_final = pd.merge(df_final, disponibilidad_real, how = join_type, on = 'fecha')
df_final = pd.merge(df_final, generacion, how = join_type, on = 'fecha')
df_final = pd.merge(df_final, aportes, how = join_type, on = 'fecha')
df_final = pd.merge(df_final, precio_oferta, how = join_type, on = 'fecha')
df_final = pd.merge(df_final, precio_escasez, how = join_type, on = 'fecha')
# df_final = pd.merge(df_final, ventas_bolsa, how = join_type, on = 'fecha')
# df_final = pd.merge(df_final, ventas_contrato, how = join_type, on = 'fecha')
df_final = pd.merge(df_final, vol_util, how = join_type, on = 'fecha')
df_final = pd.merge(df_final, indice_ONI, how = join_type, on = 'fecha')
df_final = pd.merge(df_final, vertimientos, how = 'left', on = 'fecha')
df_final = pd.merge(df_final, ipc, how = join_type, on = 'fecha')
df_final = pd.merge(df_final, ipp, how = join_type, on = 'fecha')
df_final = pd.merge(df_final, calendario, how = join_type, on = 'fecha')
# Recorte al periodo de estudio (ver la configuración inicial): los datos
# crudos pueden llegar más lejos porque la extracción es automática.
fechas_final = pd.to_datetime(df_final['fecha'])
df_final = df_final[(fechas_final >= FECHA_INICIO_ESTUDIO) & (fechas_final <= FECHA_FIN_ESTUDIO)].reset_index(drop=True)
# Conjunto consolidado: entrada del análisis exploratorio, de la selección de variables
# y de todos los notebooks de modelado.
df_final.to_csv(os.path.join(abspath,"datasets", "procesado","1_dataset_unificado.csv"), index=False) # .H5
df_final.head(5)

,fecha,precio_bolsa,demanda,disp_dec_hidraulica,disp_dec_no_termica,disp_dec_termica,disponibilidad_declarada,gen_hidraulica,gen_no_termica,gen_termica,...,ipp_mineria_nivel,ipp_importados_nivel,ipp_general_nivel,ipp_mineria_variacion_interanual,ipp_importados_variacion_interanual,ipp_general_variacion_interanual,dia_semana,dia_mes,mes,tipo_dia
0,2010-01-01,120.501228,1.133616e+08,188034000.0,4778000.0,89259000.0,282071000.0,35796804.67,6020226.83,73602052.96,...,97.28,91.08,89.52,23.15483,-5.733803,0.482658,4,1,1,3
1,2010-01-02,118.495340,1.254718e+08,186794000.0,4778000.0,89274000.0,280846000.0,46595646.99,6050741.63,75034806.16,...,97.28,91.08,89.52,23.15483,-5.733803,0.482658,5,2,1,1
2,2010-01-03,121.160145,1.211452e+08,188274000.0,4778000.0,87810000.0,280862000.0,42578875.20,5702074.83,75356828.41,...,97.28,91.08,89.52,23.15483,-5.733803,0.482658,6,3,1,2
3,2010-01-04,123.962937,1.439317e+08,180374000.0,4778000.0,85899000.0,271051000.0,62242326.79,5815643.95,77899488.25,...,97.28,91.08,89.52,23.15483,-5.733803,0.482658,0,4,1,0
4,2010-01-05,120.666437,1.498691e+08,182754000.0,4778000.0,85900000.0,273432000.0,69173462.65,7172419.15,75802147.29,...,97.28,91.08,89.52,23.15483,-5.733803,0.482658,1,5,1,0


In [21]:
# Verificación final: confirma cuántas filas y columnas quedó df_final tras
# el merge — debe coincidir con lo esperado (5964 filas, 30 columnas para
# el rango 2010-01-01 a 2026-04-30: las 25 columnas previas de ONI/IPC/
# IPP/precio_escasez + las 4 nuevas variables de calendario, dia_semana,
# dia_mes, mes y tipo_dia + indice_ONI_publicado, añadida el 2026-09-21).
print("Dimensiones", df_final.shape)

Dimensiones (5964, 30)
